In [ ]:
import sys

sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/utils")
sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/database")

In [ ]:
# Base packages -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import logging
import re
import ast
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook

# Paraellel Processing -----------------------------------------------------------------------
import dask
from dask import distributed, dataframe as dd
from dask.diagnostics import ProgressBar

# Model work --------------------------------------------------------------------------------
from transformers import AutoModel, AutoTokenizer
import torch
from glob import glob


In [ ]:
# hide warnings

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Disable TensorFlow INFO and WARNING messages
import tensorflow as tf
tf.get_logger().setLevel('ERROR')  # Disable TensorFlow ERROR messages (optional)

import warnings
# Filter out Dask client warnings
warnings.filterwarnings("ignore")


In [ ]:
import chromaDB as cbd # specter_ef, create_text_splitter, create_chroma_client
from db_tools import db_connection
from utils import import_config

config, keys = import_config()

# Chroma setup --------------------------------------------------------------------------------
chroma_server_host = "3.88.178.230"

chroma_client = cbd.create_chroma_client(chroma_server_host)

# Modelish stuff --------------------------------------------------------------------------------

model = AutoModel.from_pretrained("allenai/specter")
tokenizer = AutoTokenizer.from_pretrained("allenai/specter")

# Create embeddings function with specter model
specter_embeder = cbd.specter_ef(model, tokenizer)

specter_embeder.create_text_splitter()

collection = chroma_client.get_or_create_collection(
        "fulltext_specter", embedding_function=specter_embeder)
collection.count()

In [ ]:
def remove_exisiting(df, collection):
    """ Remove existing ids from dataframe """
    # get unique ids from collection that are the corpus ids
    query_filter = {
        "$or": [{'corpusid': str(c)} for c in df['corpusid'].unique()]
    }

    # get ids in df from collection
    temp = collection.get(
        include=["metadatas"],
        where=query_filter
    )

    if len(temp['ids']) > 0: 
        test_ids = pd.DataFrame(temp['metadatas'])

        existing_ids = test_ids.astype(str)['corpusid'].unique()

        df = df[~df['corpusid'].astype(str).isin(existing_ids)]
        
    return df

def create_split_text(df):

    df['documents'] = df.apply(
        lambda x: specter_embeder.split_text(x['documents']), axis=1)

    return df


def create_ids(df):
    """ Create ids for each chunk """

    df['ids'] = df.apply(lambda x: [
                         f"{x['corpusid']}-{i}" for i in range(len(x['documents']))], axis=1)

    return df


def create_metadatas(df):
    """ Create metadatas for each chunk """

    df['metadatas'] = df.apply(lambda x: [{'corpusid': x['corpusid'], 'chunk':i} for i in range(
        len(x['documents']))], axis=1)

    return df


def create_embeddings(df):

    df['embeddings'] = df.apply(
        lambda x: specter_embeder.embed_documents(x['documents']), axis=1)

    return df

In [ ]:
file_list = glob('/home/ubuntu/work/data/fulltext_parquets/*.parquet')

In [ ]:
file_list = glob('/home/ubuntu/work/data/fulltext_parquets/*.parquet')

for f in tqdm(file_list[2:3], desc="Processing files"):
    df = pd.read_parquet(f)
    df = df.rename({'text':'documents'}, axis = 1)
    df = df.dropna(subset=['documents'])
    
    # Add documents to collection
    n_partitions = len(df) // 10

    # print(n_partitions)

    df_split = np.array_split(df, n_partitions)

    for i in tqdm(range(len(df_split)), desc="----- partitions"):
        df_split[i] = remove_exisiting(df_split[i], collection)
        if len(df_split[i]) > 0:
            df_temp = create_split_text(df_split[i])
            df_temp = create_embeddings(df_temp)
            df_temp = create_ids(df_temp)
            df_temp = create_metadatas(df_temp)
            df_temp = df_temp.drop(columns=['corpusid'])
            dict_series = df_temp.to_dict('records')

            temp_dict = {
                'documents': [],
                'ids': [],
                'embeddings': [],
                'metadatas': []
            }

            # combine dictionaries into one
            for ds in dict_series:
                for k in temp_dict.keys():
                    temp_dict[k] = temp_dict[k] + ds[k]

            try:
                collection.add(**temp_dict)

                print('Collection count: ', collection.count())
                
                # track which files are completed
                with open('/home/ubuntu/work/therapeutic_accelerator/logs/finished_fulltext_files.txt', 'a') as file:
                    # Step 2: Write the text to append
                    file.write(f)
            except Exception as e:
                logging.exception(
                    "Error occurred while adding documents to collection")


In [ ]:
df_split[i]